In [5]:
import os
import pandas as pd
from prettytable import PrettyTable

from lib.uncertinay_rat import SimulateRat

In [6]:
DS_HOTPOTQA = '../02_data/2026-05-12_base_k10/hotpotqa'
DS_HOTPOTQA_RANDOM = '../02_data/2026-05-17_random_k10/hotpotqa'

In [7]:
def get_retrieval_success(row):
    ret_set = set([f"{r['document_id']}:{r['index']}" for r in row['retrieved']])
    ref_set = set([f"{r['document_id']}:{r['index']}" for r in row['reference']])

    return set(ref_set) <= set(ret_set)

def set_abstain(row):
    return (row['generated_answer'] == 'I DO NOT KNOW') or (row['generated_answer'] == 'NOT ENOUGH INFO')

def set_task_success(row):
    return row['correct_answer']

def set_generator_success(row):
    if row['retriever_success'] == True:
        return row['task_success']
    else:
        return row['abstain']

In [8]:
results_all = {}
success_rates = {}
samples_all = {}

simulate = SimulateRat()

def load(base, g, experiment, random=False):
    if experiment == '.DS_Store':
        return
    if g != 'qwen':
        return
    if experiment in ['hybrid', 'oracle', 'empty']:
        return

    results = pd.read_json(f'{base}/{g}/{experiment}/results.json')
    dataset = base.split('/')[-1]

    results['dataset'] = dataset
    results['generator'] = g
    results['retriever_strategy'] = experiment
    results['random'] = random

    results['correct_query'] = True
    results['retriever_success'] = results.apply(get_retrieval_success, axis=1)
    results['abstain'] = results.apply(set_abstain, axis=1)
    results['task_success'] = results.apply(set_task_success, axis=1)
    results['generator_success'] = results.apply(set_generator_success, axis=1)

    key = f"{dataset}_{g}_{experiment}"
    if random == True:
            key = f'{key}_random'

    results_all[key] = results
    success_rates[key], samples_all[key] = simulate.compute_uncertainty(results)

for d in [DS_HOTPOTQA]:
    for g in os.listdir(f"{d}"):
        if g == '.DS_Store':
            continue

        for experiment in os.listdir(f"{d}/{g}/"):
            load(d, g, experiment, False)

for d in [DS_HOTPOTQA_RANDOM]:
    for g in os.listdir(f"{d}"):
        if g == '.DS_Store':
            continue

        for experiment in os.listdir(f"{d}/{g}/"):
            load(d, g, experiment, True)

out = pd.concat((df for df in results_all.values()), ignore_index=True)

In [9]:
out.groupby(['dataset', 'generator', 'retriever_strategy', 'random'])[['retriever_success', 'abstain', 'task_success', 'generator_success']].mean().round(2)

retriever_success  abstain  \
dataset  generator retriever_strategy random                               
hotpotqa qwen      dense              False                0.24     0.45   
                                      True                 0.24     0.51   
                   sparse             False                0.42     0.26   
                                      True                 0.42     0.34   

                                              task_success  generator_success  
dataset  generator retriever_strategy random                                   
hotpotqa qwen      dense              False           0.29               0.59  
                                      True            0.28               0.65  
                   sparse             False           0.41               0.51  
                                      True            0.40               0.60

In [10]:
t = PrettyTable(field_names=['Dataset', 'Retriever Strategy', 'Random', 'P(R=1)', 'P(A=1)', 'P(T=1)', 'P(G=1)'])

for experiment in sorted(success_rates.keys()):
    id = experiment.split('_')
    random = len(id) > 3
    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2]) & (out['random'] == random)]

    t.add_row([
        id[0],
        id[2],
        'True' if random else 'False',
        f"{success_rates[experiment]['r']['mean']:.2f}",
        f"{success_rates[experiment]['a']['mean']:.2f}",
        f"{success_rates[experiment]['t']['mean']:.2f}",
        f"{success_rates[experiment]['g']['mean']:.2f}"
    ]) 
 
t

Dataset,Retriever Strategy,Random,P(R=1),P(A=1),P(T=1),P(G=1)
hotpotqa,dense,False,0.24,0.45,0.29,0.59
hotpotqa,dense,True,0.24,0.51,0.28,0.65
hotpotqa,sparse,False,0.42,0.26,0.41,0.51
hotpotqa,sparse,True,0.42,0.34,0.40,0.60


In [11]:
t = PrettyTable(field_names=['Dataset', 'Retriever Strategy', 'Random', 'P(A=1|R=1)', 'P(A=1|R=0)', 'P(T=1|R=1,A=0)', 'P(T=1|R=0,A=0)'])

for experiment in sorted(success_rates.keys()):
    id = experiment.split('_')
    random = len(id) > 3
    df = out.loc[(out['dataset'] == id[0]) & (out['generator'] == id[1]) & (out['retriever_strategy'] == id[2]) & (out['random'] == random)]

    t.add_row([
        id[0],
        id[2],
        'True' if random else 'False',
        f"{success_rates[experiment]['a_r1']['mean']:.2f}",
        f"{success_rates[experiment]['a_r0']['mean']:.2f}",
        f"{success_rates[experiment]['t_r1_a0']['mean']:.2f}",
        f"{success_rates[experiment]['t_r0_a0']['mean']:.2f}"
    ]) 
 
t

Dataset,Retriever Strategy,Random,P(A=1|R=1),P(A=1|R=0),"P(T=1|R=1,A=0)","P(T=1|R=0,A=0)"
hotpotqa,dense,False,0.04,0.58,0.63,0.44
hotpotqa,dense,True,0.04,0.67,0.64,0.50
hotpotqa,sparse,False,0.03,0.42,0.65,0.42
hotpotqa,sparse,True,0.03,0.57,0.67,0.51
